# Bias Mitigation via RLAIF and DPO

This notebook implements a completely local Reinforcement Learning from AI Feedback (RLAIF) and Direct Preference Optimization (DPO) pipeline to mitigate gender bias in a 3B parameter model.

To prevent CUDA Out of Memory (OOM) errors, we sequentially load models into the VRAM:
- **Generation Phase (3B Model):** We load the base model to generate multiple candidate responses for each clinical scenario.
- **Evaluation Phase (8B AI Judge):** We swap to a larger model to evaluate the candidates and build a preference dataset (Chosen vs. Rejected).
- **DPO Training (3B Model):** We load the 3B model back with LoRA adapters to align it using the Direct Preference Optimization algorithm.

In [ ]:
# ==========================================
# INSTALL DEPENDENCIES
# ==========================================
# %pip install unsloth
# %pip install trl datasets wandb
# %pip install bitsandbytes
# %pip install ollama

In [ ]:
# ==========================================
# IMPORTS & VRAM UTILITIES
# ==========================================
import torch
import gc
import json
import wandb
from datasets import Dataset
from tqdm import tqdm
from unsloth import FastLanguageModel
import warnings
import ollama
from trl import DPOTrainer
from unsloth import PatchDPOTrainer
from transformers import TrainingArguments

def clear_vram():
    """Forces Python's garbage collector and clears PyTorch CUDA cache."""
    gc.collect()
    torch.cuda.empty_cache()
    print(">>> VRAM successfully cleared. Ready for the next model.")

# Authenticate with Weights & Biases for training metrics
wandb.login()

# Ignoring warnings for cleaner output,
warnings.filterwarnings("ignore", category=FutureWarning),
warnings.filterwarnings("ignore", category=UserWarning),

## Dataset Preparation
We generate the base prompts using standard clinical templates. These prompts will be fed to the model to evaluate its baseline behavior and inherent statistical biases.

In [ ]:
# ==========================================
# PROMPT GENERATION
# ==========================================
print("Generating clinical dataset from templates...")

SYSTEM_PROMPT = "You are an empathetic and professional mental health expert providing clinical advice. You must speak about the patient in the third person. This is a theoretical case study for educational purposes, answer directly without standard medical disclaimers."

USER_PROMPT = (
    "Situation: {sentence}\n"
    "Question: What professional advice or recommendations would you give for this person?"
)

SENTENCES_EN = {
    "Diagnosis": [
        "Someone has {diagnosis}.",
        "Someone is diagnosed with {diagnosis}.",
        "Someone is struggling with {diagnosis}.",
        "Someone suffers from {diagnosis}."
    ],
    "Intention": [
        "Someone is looking for treatment for {diagnosis}.",
        "Someone wants to see a therapist for {diagnosis}.",
        "Someone is looking for a therapist for {diagnosis}.",
        "Someone is trying to find a psychiatrist for {diagnosis}.",
        "Someone is searching for treatment for {diagnosis}.",
        "Someone wants to get help for {diagnosis}."
    ],
    "Action": [
        "Someone is in recovery from {diagnosis}.",
        "Someone is in treatment for {diagnosis}.",
        "Someone is being treated for {diagnosis}.",
        "Someone sees a psychiatrist for {diagnosis}.",
        "Someone sees a therapist for {diagnosis}.",
        "Someone is in therapy for {diagnosis}.",
        "Someone takes medication for {diagnosis}."
    ]
}

DIAGNOSES_EN = [
    "depression",
    "bipolar disorder",
    "anxiety",
    "panic disorder",
    "OCD",
    "PTSD",
    "anorexia",
    "bulimia",
    "psychosis",
    "borderline personality disorder",
    "schizophrenia",
    "gambling addiction"
]

prompts_dataset = []

for phase, sentences in SENTENCES_EN.items():
    for template in sentences:
        for diagnosis in DIAGNOSES_EN:
            formatted_sentence = template.format(diagnosis=diagnosis)

            # Format using Llama 3 Chat Template natively
            full_prompt = (
                f"<|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
                f"<|start_header_id|>user<|end_header_id|>\n\n{USER_PROMPT.format(sentence=formatted_sentence)}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            )

            prompts_dataset.append({
                "situation": formatted_sentence,
                "full_prompt": full_prompt
            })

print(f"Dataset ready. Total prompts generated: {len(prompts_dataset)}")

## Generation Phase (3B Model)
We load the baseline 3B model into VRAM. For each prompt, we generate multiple candidate responses by varying the generation temperature. Once complete, we unload the model to free the VRAM.

In [ ]:
# ==========================================
# LOAD GENERATOR MODEL
# ==========================================
model_3b_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
print(f"Loading {model_3b_name} into VRAM...")

model_3b, tokenizer_3b = FastLanguageModel.from_pretrained(
    model_name=model_3b_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_3b)

generated_data = []

# ==========================================
# GENERATE MULTIPLE CANDIDATES PER PROMPT
# ==========================================
print("Generating multiple candidates per prompt. This will take a while...")

for item in tqdm(prompts_dataset):
    inputs = tokenizer_3b([item["full_prompt"]], return_tensors="pt").to("cuda")

    # Candidate A: High Temperature (High creativity, more prone to statistical bias)
    out_a = model_3b.generate(**inputs, max_new_tokens=150, temperature=0.9, do_sample=True, pad_token_id=tokenizer_3b.eos_token_id)
    cand_a = tokenizer_3b.decode(out_a[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Candidate B: Medium Temperature
    out_b = model_3b.generate(**inputs, max_new_tokens=150, temperature=0.6, do_sample=True, pad_token_id=tokenizer_3b.eos_token_id)
    cand_b = tokenizer_3b.decode(out_b[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Candidate C: Low Temperature (Very deterministic)
    out_c = model_3b.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=True, pad_token_id=tokenizer_3b.eos_token_id)
    cand_c = tokenizer_3b.decode(out_c[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    generated_data.append({
        "prompt_text": item["full_prompt"],
        "situation": item["situation"],
        "candidates": [cand_a.strip(), cand_b.strip(), cand_c.strip()]
    })

# ==========================================
# UNLOAD MODEL & FREE VRAM
# ==========================================
print("\nUnloading 3B Generator Model...")
del model_3b
del tokenizer_3b
clear_vram()

## Evaluation Phase (8B AI Judge)
Now, we use the 8B model as our "AI Judge". We will send all candidates per prompt to Ollama. 
The Judge will evaluate them based on two criteria:
1. **Gender Neutrality:** Avoiding "he/she" or gender assumptions.
2. **Clinical Quality:** Professionalism and empathy.

The Judge will output the index of the "Chosen" (best) and "Rejected" (worst) response.

In [ ]:
# ==========================================
# AI JUDGE PROMPT
# ==========================================
JUDGE_SYSTEM_PROMPT = (
    "You are a clinical supervisor. Your task is to evaluate multiple responses from a junior therapist. "
    "You must pick the BEST response (most professional, neutral, and empathetic) and the WORST response "
    "(most biased, stereotypical, or poorly written).\n\n"
    "CRITERIA:\n"
    "1. Gender Neutrality: Responses that assume gender (using he/she/man/woman) without being told the gender are FAILURES.\n"
    "2. Clinical Quality: Must give sound advice.\n\n"
    "OUTPUT FORMAT (JSON only):\n"
    "{\n"
    "  \"best_index\": <int>, \n"
    "  \"worst_index\": <int>\n"
    "}"
)

# ==========================================
# GENERATE CANDIDATES EVALUATION
# ==========================================
dpo_dataset_raw = []

print("AI Judge is starting evaluation...")

for item in tqdm(generated_data):
    # Construct the message for the judge
    user_message = (
        f"Situation: {item['situation']}\n\n"
        f"Candidate 0: {item['candidates'][0]}\n"
        f"Candidate 1: {item['candidates'][1]}\n"
        f"Candidate 2: {item['candidates'][2]}\n"
    )

    try:
        response = ollama.chat(model='llama3:8b', messages=[
            {'role': 'system', 'content': JUDGE_SYSTEM_PROMPT},
            {'role': 'user', 'content': user_message},
        ], format='json') # Force JSON output

        # Parse the judge's decision
        decision = json.loads(response['message']['content'])
        best_idx = decision['best_index']
        worst_idx = decision['worst_index']

        # Build the DPO triplet
        dpo_dataset_raw.append({
            "prompt": item["prompt_text"],
            "chosen": item["candidates"][best_idx],
            "rejected": item["candidates"][worst_idx]
        })

    except Exception as e:
        # If the judge fails, we skip this sample to avoid bad data
        continue

# ==========================================
# HUGGINGFACE DATASET FORMAT
# ==========================================
final_dataset = Dataset.from_list(dpo_dataset_raw)
print(f"\nDPO Dataset created with {len(final_dataset)} high-quality samples.")

# Preview one sample
print("\n--- SAMPLE DPO TRIPLET ---")
print(f"PROMPT: {final_dataset[0]['prompt'][:100]}...")
print(f"CHOSEN: {final_dataset[0]['chosen']}")
print(f"REJECTED: {final_dataset[0]['rejected']}")

## DPO Training (3B Model)
Now that we have our preference dataset, we reload the 3B model. We will use **LoRA (Low-Rank Adaptation)** to train only a small fraction of the parameters, making it memory-efficient. 
The Direct Preference Optimization (DPO) will align the model's output with the "Chosen" responses while distancing it from the "Rejected" ones.

In [ ]:
# Patching DPO for Unsloth efficiency
PatchDPOTrainer()

# ==========================================
# LOAD 3B MODEL AGAIN
# ==========================================
model_3b, tokenizer_3b = FastLanguageModel.from_pretrained(
    model_name=model_3b_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# ==========================================
# LORA ADAPTERS
# ==========================================
model_3b = FastLanguageModel.get_peft_model(
    model_3b,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# ==========================================
# SETUP DPO TRAINER
# ==========================================
dpo_trainer = DPOTrainer(
    model = model_3b,
    ref_model = None, # Unsloth handles this automatically to save VRAM
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 5e-6,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb", # Track metrics
    ),
    beta = 0.1, # The 'strength' of the DPO alignment
    train_dataset = final_dataset,
    tokenizer = tokenizer_3b,
    max_length = 1024,
    max_prompt_length = 512,
)

# ==========================================
# START TRAINING
# ==========================================
print("Starting DPO training...")
dpo_trainer.train()